# BSRBF-KAN + KAN-Mixer + bộ hiệu chỉnh theo tầng

Notebook v7 dùng 3 checkpoint v5. V6 có lỗi quy trình: sau khi huấn luyện đầu phân loại tầng, việc khôi phục checkpoint tọa độ tốt nhất đã xóa luôn trọng số phân loại tầng. V7 huấn luyện **đầu tầng riêng**, giữ checkpoint tầng tốt nhất, rồi huấn luyện bộ hiệu chỉnh tọa độ theo xác suất tầng. Tầng lúc suy luận là **tầng dự đoán**, không dùng nhãn thật.

Chạy từ trên xuống trên Colab GPU. Upload `v5_checkpoints.zip` khi được hỏi. Dữ liệu UJI được tải tự động nếu chưa có. Validation giữ lại những vị trí đo chưa xuất hiện trong train; official test được đánh giá sau khi chọn checkpoint. Mốc **<5 m chưa được bảo đảm**. Official test UJI đã được xem trong v4–v6, nên kết quả v7 là thử nghiệm tiếp nối, không phải test mù mới.


In [ ]:
import os
import math
import copy
import random
import shutil
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
from pathlib import Path
import json

from pathlib import Path
from io import BytesIO
import json


In [ ]:
# Cấu hình
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GRID_SIZE = 4
SPLINE_ORDER = 3
MIXER_BLOCKS = 2
BATCH_SIZE = 64
WARMUP_EPOCHS = 12
JOINT_EPOCHS = 55
PATIENCE = 12
VAL_SIZE = 0.15
SAVE_DIR = Path('/content/floor_adapter_v7')
CHECKPOINT_DIR = Path('/content/v5_checkpoints')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Thiết bị:', DEVICE, '| Kết quả:', SAVE_DIR)


In [ ]:
# Nạp 3 checkpoint v5. Nếu đã đặt sẵn trong CHECKPOINT_DIR, bỏ qua upload.
paths = sorted(CHECKPOINT_DIR.glob('building_*_bsrbf_residual_mixer.pt'))
if len(paths) != 3:
    from google.colab import files
    uploaded = files.upload()  # Chọn v5_checkpoints.zip hoặc 3 tệp .pt.
    for name, payload in uploaded.items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(BytesIO(payload)) as archive:
                for member in archive.namelist():
                    short = Path(member).name
                    if short.startswith('building_') and short.endswith('_bsrbf_residual_mixer.pt'):
                        (CHECKPOINT_DIR/short).write_bytes(archive.read(member))
        elif name.startswith('building_') and name.endswith('_bsrbf_residual_mixer.pt'):
            (CHECKPOINT_DIR/Path(name).name).write_bytes(payload)
paths = sorted(CHECKPOINT_DIR.glob('building_*_bsrbf_residual_mixer.pt'))
assert len(paths) == 3, f'Cần 3 checkpoint; hiện có {paths}'
print([p.name for p in paths])


In [ ]:
class RadialBasisFunction(nn.Module):

    def __init__(
        self,
        grid_min=-1.5,
        grid_max=1.5,
        num_grids=8,
        denominator=None
    ):
        super().__init__()

        grid = torch.linspace(
            grid_min,
            grid_max,
            num_grids
        )

        self.grid = nn.Parameter(
            grid,
            requires_grad=False
        )

        self.denominator = (
            denominator
            or
            (grid_max - grid_min) / (num_grids - 1)
        )

    def forward(self, x):

        return torch.exp(
            -(
                (
                    x[..., None]
                    - self.grid
                )
                / self.denominator
            ) ** 2
        )

In [ ]:
class BSRBF_KANLayer(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        grid_size=5,
        spline_order=3,
        base_activation=nn.SiLU,
        grid_range=(-1.5, 1.5)
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim

        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layernorm = nn.LayerNorm(
            input_dim
        )

        self.base_activation = (
            base_activation()
        )

        # ================================================================
        # Base branch
        # ================================================================

        self.base_weight = nn.Parameter(
            torch.empty(
                output_dim,
                input_dim
            )
        )

        nn.init.kaiming_uniform_(
            self.base_weight,
            a=math.sqrt(5)
        )

        # ================================================================
        # BSRBF branch
        # ================================================================

        self.spline_weight = nn.Parameter(
            torch.empty(
                output_dim,
                input_dim * (
                    grid_size
                    + spline_order
                )
            )
        )

        nn.init.kaiming_uniform_(
            self.spline_weight,
            a=math.sqrt(5)
        )

        self.rbf = RadialBasisFunction(
            grid_min=grid_range[0],
            grid_max=grid_range[1],
            num_grids=(
                grid_size
                + spline_order
            )
        )

        h = (
            grid_range[1]
            - grid_range[0]
        ) / grid_size

        grid = (
            (
                torch.arange(
                    -spline_order,
                    grid_size
                    + spline_order
                    + 1
                )
                * h
                + grid_range[0]
            )
            .expand(
                input_dim,
                -1
            )
            .contiguous()
        )

        self.register_buffer(
            "grid",
            grid
        )

    # =====================================================================
    # B-SPLINE
    # =====================================================================

    def b_splines(self, x):

        assert x.dim() == 2
        assert x.size(1) == self.input_dim

        grid = self.grid

        x = x.unsqueeze(-1)

        bases = (
            (
                x >= grid[:, :-1]
            )
            &
            (
                x < grid[:, 1:]
            )
        ).to(x.dtype)

        for k in range(
            1,
            self.spline_order + 1
        ):

            bases = (
                (
                    (
                        x
                        - grid[:, :-(k + 1)]
                    )
                    /
                    (
                        grid[:, k:-1]
                        - grid[:, :-(k + 1)]
                    )
                )
                * bases[:, :, :-1]
            ) + (
                (
                    (
                        grid[:, k + 1:]
                        - x
                    )
                    /
                    (
                        grid[:, k + 1:]
                        - grid[:, 1:-k]
                    )
                )
                * bases[:, :, 1:]
            )

        return bases.contiguous()

    # =====================================================================
    # FORWARD
    # =====================================================================

    def forward(self, x):

        x = self.layernorm(x)

        # Base branch
        base_output = F.linear(
            self.base_activation(x),
            self.base_weight
        )

        # B-spline branch
        bs_output = (
            self
            .b_splines(x)
            .reshape(
                x.size(0),
                -1
            )
        )

        # RBF branch
        rbf_output = (
            self
            .rbf(x)
            .reshape(
                x.size(0),
                -1
            )
        )

        # Combined BSRBF
        bsrbf_output = (
            bs_output
            + rbf_output
        )

        bsrbf_output = F.linear(
            bsrbf_output,
            self.spline_weight
        )

        return (
            base_output
            + bsrbf_output
        )

In [ ]:
class BSRBF_KAN(nn.Module):

    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        base_activation=nn.SiLU
    ):
        super().__init__()

        self.layers = nn.ModuleList()

        for input_dim, output_dim in zip(
            layers_hidden,
            layers_hidden[1:]
        ):

            self.layers.append(
                BSRBF_KANLayer(
                    input_dim,
                    output_dim,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    base_activation=base_activation
                )
            )

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


print("BSRBF-KAN model loaded.")

In [ ]:
BASE_DIR = "/content/UJIndoorLoc"  # đặt CSV vào đây nếu đã tải trước

TRAIN_PATH = os.path.join(
    BASE_DIR,
    "trainingData.csv"
)

TEST_PATH = os.path.join(
    BASE_DIR,
    "validationData.csv"
)

if not (
    os.path.exists(TRAIN_PATH)
    and
    os.path.exists(TEST_PATH)
):

    print("\nDownloading UJIIndoorLoc...")

    zip_path = "/content/uji.zip"

    url = (
        "https://archive.ics.uci.edu/"
        "static/public/310/"
        "ujiindoorloc.zip"
    )

    urllib.request.urlretrieve(
        url,
        zip_path
    )

    temp_dir = "/content/UJI_TEMP"

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        z.extractall(
            temp_dir
        )

    found_train = None
    found_test = None

    for root, dirs, files in os.walk(
        temp_dir
    ):

        for f in files:

            if f.lower() == "trainingdata.csv":

                found_train = os.path.join(
                    root,
                    f
                )

            elif f.lower() == "validationdata.csv":

                found_test = os.path.join(
                    root,
                    f
                )

    if found_train is None or found_test is None:

        raise FileNotFoundError(
            "UJIIndoorLoc CSV files not found."
        )

    os.makedirs(
        BASE_DIR,
        exist_ok=True
    )

    shutil.copy(
        found_train,
        TRAIN_PATH
    )

    shutil.copy(
        found_test,
        TEST_PATH
    )

print("\nTraining file :", TRAIN_PATH)
print("Test file     :", TEST_PATH)

In [ ]:
train_df = pd.read_csv(
    TRAIN_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

WAP_COLS = [
    c
    for c in train_df.columns
    if c.upper().startswith("WAP")
]

print("\n" + "=" * 90)
print("DATASET")
print("=" * 90)

print(
    "Training samples:",
    len(train_df)
)

print(
    "Official test  :",
    len(test_df)
)

print(
    "WAPs           :",
    len(WAP_COLS)
)

print(
    "Buildings      :",
    sorted(
        train_df.BUILDINGID.unique()
    )
)

print("=" * 90)

In [ ]:
def preprocess_rss(df):

    X = (
        df[WAP_COLS]
        .to_numpy(
            dtype=np.float32
        )
    )

    # Missing AP
    X[X == 100] = -110.0

    # Keep realistic RSS interval
    X = np.clip(
        X,
        -110.0,
        -30.0
    )

    # [-110,-30] -> [-1,+1]
    X = (
        2.0
        *
        (
            (X + 110.0)
            /
            80.0
        )
        -
        1.0
    )

    return X.astype(
        np.float32
    )


print("RSS preprocessing ready.")

In [ ]:
class LocalizationLoss(nn.Module):

    def __init__(
        self,
        alpha=0.25
    ):
        super().__init__()

        self.alpha = alpha

    def forward(
        self,
        pred,
        target
    ):

        dist = torch.sqrt(
            torch.sum(
                (
                    pred
                    - target
                ) ** 2,
                dim=1
            )
            + 1e-8
        )

        euclidean_loss = (
            dist.mean()
        )

        mse_loss = F.mse_loss(
            pred,
            target
        )

        return (
            euclidean_loss
            +
            self.alpha
            * mse_loss
        )


print("LocalizationLoss ready.")

In [ ]:
# Residual KAN-Mixer trên đặc trưng của BSRBF-KAN đã học.
class SmallKANMixerBlock(nn.Module):
    def __init__(self, tokens=16, channels=8):
        super().__init__()
        self.token_norm = nn.LayerNorm(channels)
        self.token_net = nn.Sequential(
            BSRBF_KANLayer(tokens, 32, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER),
            BSRBF_KANLayer(32, tokens, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER))
        self.channel_norm = nn.LayerNorm(channels)
        self.channel_net = nn.Sequential(
            BSRBF_KANLayer(channels, 16, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER),
            BSRBF_KANLayer(16, channels, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER))

    def forward(self, x):
        batch, tokens, channels = x.shape
        t = self.token_norm(x).transpose(1, 2).reshape(-1, tokens)
        x = x + self.token_net(t).reshape(batch, channels, tokens).transpose(1, 2)
        c = self.channel_norm(x).reshape(-1, channels)
        return x + self.channel_net(c).reshape(batch, tokens, channels)


class ResidualKANMixer(nn.Module):
    def __init__(self, n_blocks=MIXER_BLOCKS):
        super().__init__()
        self.rss_projection = nn.Sequential(nn.Linear(520, 64), nn.LayerNorm(64))
        self.blocks = nn.Sequential(*[SmallKANMixerBlock() for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(128), nn.Linear(128, 64),
                                  nn.SiLU(), nn.Linear(64, 2))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, rssi, features):
        tokens = torch.cat([features, self.rss_projection(rssi)], dim=1).reshape(-1, 16, 8)
        return self.head(self.blocks(tokens).flatten(1))




In [ ]:
# Dữ liệu cho v6: augmentation chỉ dùng ở train.
class WiFiDataset(torch.utils.data.Dataset):
    def __init__(self, frame, scaler_mean, scaler_scale, floor_to_id, augment=False):
        self.rss = torch.from_numpy(preprocess_rss(frame))
        self.xy_world = torch.from_numpy(frame[['LONGITUDE','LATITUDE']].to_numpy(np.float64))
        self.xy = ((self.xy_world - torch.as_tensor(scaler_mean,dtype=torch.float64)) /
                   torch.as_tensor(scaler_scale,dtype=torch.float64)).float()
        self.floor = torch.tensor([floor_to_id[int(x)] for x in frame.FLOOR],dtype=torch.long)
        self.augment = augment
    def __len__(self): return len(self.rss)
    def __getitem__(self, index):
        x = self.rss[index].clone()
        if self.augment:
            observed = x > -0.999
            gain = torch.randn(()) * 0.10
            noise = torch.randn_like(x) * 0.05
            x = torch.where(observed, (x+gain+noise).clamp(-1,1), x)
            x[(torch.rand_like(x)<0.06)&observed] = -1.0
        return x,self.xy[index],self.xy_world[index],self.floor[index]

def split_building(building):
    a = train_df[train_df.BUILDINGID==building].reset_index(drop=True)
    t = test_df[test_df.BUILDINGID==building].reset_index(drop=True)
    rp = (a.FLOOR.astype(str)+'_'+a.LONGITUDE.round(3).astype(str)+'_'+a.LATITUDE.round(3).astype(str)).to_numpy()
    ti,vi = next(GroupShuffleSplit(n_splits=1,test_size=VAL_SIZE,random_state=SEED).split(a,groups=rp))
    assert set(rp[ti]).isdisjoint(set(rp[vi]))
    return a.iloc[ti].reset_index(drop=True),a.iloc[vi].reset_index(drop=True),t


In [ ]:
# Khối mới giữ các trọng số Mixer v5 và thêm nhánh tầng + embedding vị trí.
class JointFloorKANMixer(nn.Module):
    def __init__(self, baseline, n_floors, alpha):
        super().__init__()
        self.baseline = baseline
        self.alpha = float(alpha)
        self.rss_projection = nn.Sequential(nn.Linear(520,64),nn.LayerNorm(64))
        self.blocks = nn.Sequential(*[SmallKANMixerBlock() for _ in range(MIXER_BLOCKS)])
        self.head = nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,2))
        self.floor_head = nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,n_floors))
        self.floor_delta = nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,2*n_floors))
        self.metric_head = nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,32))
        nn.init.zeros_(self.floor_delta[-1].weight)
        nn.init.zeros_(self.floor_delta[-1].bias)
        self.n_floors=n_floors
    def forward(self,rss):
        h=rss
        for layer in self.baseline.layers[:-1]: h=layer(h)
        base=self.baseline.layers[-1](h)
        tokens=torch.cat([h,self.rss_projection(rss)],dim=1).reshape(-1,16,8)
        z=self.blocks(tokens).flatten(1)
        floor_logits=self.floor_head(z)
        floor_probs=floor_logits.softmax(-1)
        floor_delta=self.floor_delta(z).reshape(-1,self.n_floors,2)
        weighted_delta=(floor_probs.unsqueeze(-1)*floor_delta).sum(dim=1)
        pred=base+self.alpha*self.head(z)+weighted_delta
        embedding=F.normalize(self.metric_head(z),dim=1)
        return pred,floor_logits,embedding

def metric_loss(embedding,xy_world,floor):
    with torch.no_grad():
        physical=torch.cdist(xy_world,xy_world)
        same=floor[:,None]==floor[None,:]
        eye=torch.eye(len(floor),dtype=torch.bool,device=floor.device)
        near=(physical<5)&same&~eye
        far=((physical>20)|~same)&~eye
    latent=torch.cdist(embedding,embedding)
    near_loss=latent[near].mean() if near.any() else latent.new_zeros(())
    far_loss=F.relu(1.0-latent[far]).mean() if far.any() else latent.new_zeros(())
    return near_loss+far_loss

def training_loss(outputs,xy,xy_world,floor,scale):
    pred,fl,embed=outputs
    coord_m=torch.linalg.vector_norm((pred-xy)*scale,dim=1).mean()
    floor_ce=F.cross_entropy(fl,floor)
    metric=metric_loss(embed,xy_world,floor)
    return coord_m+0.5*floor_ce+0.25*metric+0.1*F.mse_loss(pred,xy), (coord_m.item(),floor_ce.item(),metric.item())

@torch.no_grad()
def evaluate_model(model,loader,mean,scale):
    model.eval(); errors=[]; floor_ok=0; n=0
    for rss,xy,world,floor in loader:
        rss=rss.to(DEVICE)
        pred,fl,_=model(rss)
        pred_world=pred.cpu().double()*scale.double()+mean.double()
        errors.extend(torch.linalg.vector_norm(pred_world-world.double(),dim=1).numpy())
        floor_ok+=(fl.cpu().argmax(1)==floor).sum().item();n+=len(floor)
    return {'mean_m':float(np.mean(errors)),'median_m':float(np.median(errors)),
            'p90_m':float(np.percentile(errors,90)),'floor_accuracy':floor_ok/n,
            'errors':np.asarray(errors)}


In [ ]:
# Ba giai đoạn: (1) học tầng, (2) hiệu chỉnh theo tầng, (3) tinh chỉnh nhẹ BSRBF-KAN.
def load_checkpoint(path):
    safe=[np._core.multiarray._reconstruct,np.ndarray,np.dtype,type(np.dtype('float64'))]
    with torch.serialization.safe_globals(safe):
        return torch.load(path,map_location='cpu',weights_only=True)

@torch.no_grad()
def evaluate_floor(model,loader):
    model.eval(); n=0; correct=0; total_loss=0.0
    for rss,_,_,floor in loader:
        logits=model(rss.to(DEVICE))[1]
        correct+=(logits.argmax(1).cpu()==floor).sum().item()
        total_loss+=F.cross_entropy(logits,floor.to(DEVICE),reduction='sum').item()
        n+=len(floor)
    return {'accuracy':correct/n,'ce':total_loss/n}

def train_building_v7(building):
    cp=load_checkpoint(CHECKPOINT_DIR/f'building_{building}_bsrbf_residual_mixer.pt')
    tr,va,te=split_building(building)
    floors=sorted(train_df.loc[train_df.BUILDINGID==building,'FLOOR'].unique().tolist())
    floor_to_id={int(v):i for i,v in enumerate(floors)}
    mean=torch.as_tensor(cp['scaler_mean'],dtype=torch.float64)
    scale=torch.as_tensor(cp['scaler_scale'],dtype=torch.float64)
    tr_loader=DataLoader(WiFiDataset(tr,mean,scale,floor_to_id,augment=True),
                         batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    va_loader=DataLoader(WiFiDataset(va,mean,scale,floor_to_id),
                         batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    te_loader=DataLoader(WiFiDataset(te,mean,scale,floor_to_id),
                         batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    baseline=BSRBF_KAN([520,512,512,256,256,128,128,64,64,2],
                       grid_size=GRID_SIZE,spline_order=SPLINE_ORDER)
    baseline.load_state_dict(cp['baseline_state'])
    model=JointFloorKANMixer(baseline,len(floors),cp['alpha'])
    if cp['mixer_state'] is not None:
        missing,unexpected=model.load_state_dict(cp['mixer_state'],strict=False)
        assert not unexpected
    model=model.to(DEVICE)
    original=evaluate_model(model,va_loader,mean,scale)
    assert abs(original['mean_m']-cp['mixer_val_m'])<0.02
    print(f'Tòa {building}: v5 val {original["mean_m"]:.3f} m | '
          f'train/val/test {len(tr)}/{len(va)}/{len(te)}')

    # Giai đoạn 1: chỉ đầu tầng được cập nhật. Dự đoán tọa độ giữ nguyên v5.
    for p in model.parameters():p.requires_grad=False
    for p in model.floor_head.parameters():p.requires_grad=True
    opt=torch.optim.AdamW(model.floor_head.parameters(),lr=5e-4,weight_decay=1e-4)
    best_floor=0.0;best_ce=float('inf');floor_state=None;stale=0
    for epoch in range(1,26):
        model.train();total=0.0
        for rss,_,_,floor in tr_loader:
            rss,floor=rss.to(DEVICE),floor.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce=F.cross_entropy(model(rss)[1],floor)
            ce.backward()
            nn.utils.clip_grad_norm_(model.floor_head.parameters(),5.0)
            opt.step();total+=ce.item()*len(rss)
        val_floor=evaluate_floor(model,va_loader)
        if (val_floor['accuracy']>best_floor+1e-4 or
            (abs(val_floor['accuracy']-best_floor)<1e-4 and val_floor['ce']<best_ce)):
            best_floor=val_floor['accuracy'];best_ce=val_floor['ce']
            floor_state=copy.deepcopy(model.floor_head.state_dict());stale=0
        else:stale+=1
        if epoch==1 or epoch%5==0:
            print(f'Tòa {building} floor {epoch:03d} | train CE {total/len(tr):.3f} | '
                  f'val acc {val_floor["accuracy"]:.1%}')
        if stale>=8:break
    model.floor_head.load_state_dict(floor_state)
    fixed_floor=evaluate_floor(model,va_loader)
    initial=evaluate_model(model,va_loader,mean,scale)
    assert abs(initial['mean_m']-original['mean_m'])<0.02
    print(f'Tòa {building}: giữ floor val {fixed_floor["accuracy"]:.1%}; '
          f'vị trí vẫn {initial["mean_m"]:.3f} m')

    # Giai đoạn 2: chỉ floor_delta học tọa độ; mọi mạng có sẵn giữ nguyên.
    for p in model.parameters():p.requires_grad=False
    for p in model.floor_delta.parameters():p.requires_grad=True
    best_val=initial['mean_m'];best_state=copy.deepcopy(model.state_dict());best_epoch='v5'
    def coordinate_epoch(optimizer):
        model.train();total=0.0
        for rss,xy,_,_ in tr_loader:
            rss,xy=rss.to(DEVICE),xy.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred=model(rss)[0]
            loss=torch.linalg.vector_norm((pred-xy)*scale.float().to(DEVICE),dim=1).mean()
            loss.backward()
            nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),5.0)
            optimizer.step();total+=loss.item()*len(rss)
        return total/len(tr)

    opt=torch.optim.AdamW(model.floor_delta.parameters(),lr=1e-4,weight_decay=1e-5)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=5)
    stale=0
    for epoch in range(1,61):
        train_m=coordinate_epoch(opt)
        val=evaluate_model(model,va_loader,mean,scale)
        scheduler.step(val['mean_m'])
        if val['mean_m']<best_val-0.005:
            best_val=val['mean_m'];best_state=copy.deepcopy(model.state_dict())
            best_epoch=f'adapter-{epoch}';stale=0
        else:stale+=1
        if epoch==1 or epoch%5==0:
            print(f'Tòa {building} adapter {epoch:03d} | train {train_m:.3f} m | '
                  f'val {val["mean_m"]:.3f} m | floor {val["floor_accuracy"]:.1%}')
        if stale>=12:break
    model.load_state_dict(best_state)

    # Giai đoạn 3: chỉ tinh chỉnh 3 tầng BSRBF cuối và adapter, LR rất nhỏ.
    for p in model.parameters():p.requires_grad=False
    for p in model.baseline.layers[-3:].parameters():p.requires_grad=True
    for p in model.floor_delta.parameters():p.requires_grad=True
    opt=torch.optim.AdamW([
        {'params':model.baseline.layers[-3:].parameters(),'lr':5e-6},
        {'params':model.floor_delta.parameters(),'lr':2e-5}],weight_decay=1e-5)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=4)
    stale=0
    for epoch in range(1,31):
        train_m=coordinate_epoch(opt)
        val=evaluate_model(model,va_loader,mean,scale)
        scheduler.step(val['mean_m'])
        if val['mean_m']<best_val-0.005:
            best_val=val['mean_m'];best_state=copy.deepcopy(model.state_dict())
            best_epoch=f'joint-{epoch}';stale=0
        else:stale+=1
        if epoch==1 or epoch%5==0:
            print(f'Tòa {building} joint {epoch:03d} | train {train_m:.3f} m | '
                  f'val {val["mean_m"]:.3f} m | floor {val["floor_accuracy"]:.1%}')
        if stale>=10:break
    model.load_state_dict(best_state)

    # Chỉ giờ mới đánh giá official test.
    test=evaluate_model(model,te_loader,mean,scale)
    print(f'Tòa {building}: v5 {cp["mixer_test_m"]:.3f} m -> v7 {test["mean_m"]:.3f} m '
          f'| floor test {test["floor_accuracy"]:.1%} | chọn {best_epoch}')
    torch.save({'model_state':model.state_dict(),'building':building,'floors':floors,
                'scaler_mean':cp['scaler_mean'],'scaler_scale':cp['scaler_scale'],
                'alpha':cp['alpha'],'best_val_m':best_val,'test_m':test['mean_m'],
                'best_epoch':best_epoch},SAVE_DIR/f'building_{building}_v7.pt')
    return {'building':building,'n_test':len(te),'v5_test_m':cp['mixer_test_m'],
            'v7_test_m':test['mean_m'],'v5_val_m':cp['mixer_val_m'],
            'v7_val_m':best_val,'floor_accuracy':test['floor_accuracy'],
            'best_epoch':best_epoch,'errors':test['errors']}

results=[]
for building in (0,1,2):
    results.append(train_building_v7(building))
    if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
# Kết quả v7 trên official test.
comparison=pd.DataFrame([{k:v for k,v in r.items() if k!='errors'} for r in results])
display(comparison.round(4))
errors=np.concatenate([r['errors'] for r in results])
summary={'n_test':int(len(errors)),'metric':'mean Euclidean distance (m)',
         'v5_test_m':float(sum(r['n_test']*r['v5_test_m'] for r in results)/len(errors)),
         'v7_test_m':float(errors.mean()),'v7_median_m':float(np.median(errors)),
         'v7_p90_m':float(np.percentile(errors,90)),'under_5m':bool(errors.mean()<5.0)}
print(json.dumps(summary,indent=2,ensure_ascii=False))
comparison.to_csv(SAVE_DIR/'comparison.csv',index=False)
(SAVE_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
